# 01 — Dataset Reconnaissance

Enveda CASMI 2026 — Molecule ID From Mass Spectra.

This notebook reproduces every number quoted in `research/00_dataset_report.md`. It is meant
to be *reusable*: re-run top-to-bottom any time the data files change or a new question comes
up, rather than treated as disposable scratch exploration.

**Reader note (important gotcha):** `pandas.read_parquet()` / `pyarrow.parquet.read_table()`
fail on `train.parquet` / `test.parquet` in this environment with
`OSError: Repetition level histogram size mismatch` (pyarrow 19.0.0). We use `duckdb`
instead, which reads both files without issue. See `src/data/loader.py`.


In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import duckdb
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

con = duckdb.connect()
TRAIN = "../train.parquet"
TEST = "../test.parquet"
SUB = "../sample_submission.csv"


## 1. File inventory & schema

In [2]:
for name, path in [("train.parquet", TRAIN), ("test.parquet", TEST)]:
    n_rows = con.execute(f"SELECT count(*) FROM '{path}'").fetchone()[0]
    print(f"{name}: {n_rows:,} rows")


train.parquet: 2,539,608 rows
test.parquet: 1,213 rows


In [3]:
print("train.parquet schema:")
con.execute(f"DESCRIBE SELECT * FROM '{TRAIN}'").fetchdf()


train.parquet schema:


,column_name,column_type,null,key,default,extra
0,ingest_lib,VARCHAR,YES,None,None,None
1,normalized_smiles,VARCHAR,YES,None,None,None
2,inchikey,VARCHAR,YES,None,None,None
3,inchikey14,VARCHAR,YES,None,None,None
4,molecular_formula,VARCHAR,YES,None,None,None
5,ionization_mode,VARCHAR,YES,None,None,None
6,instrument_type,VARCHAR,YES,None,None,None
7,adduct,VARCHAR,YES,None,None,None
8,adduct_orig,VARCHAR,YES,None,None,None
9,precursor_mz,DOUBLE,YES,None,None,None


In [4]:
print("test.parquet schema:")
con.execute(f"DESCRIBE SELECT * FROM '{TEST}'").fetchdf()


test.parquet schema:


,column_name,column_type,null,key,default,extra
0,molecule_id,VARCHAR,YES,None,None,None
1,spectrum_id,VARCHAR,YES,None,None,None
2,ms2_mzs,DOUBLE[],YES,None,None,None
3,ms2_normalized_intensities,DOUBLE[],YES,None,None,None
4,base_peak_intensity,DOUBLE,YES,None,None,None
5,adduct,VARCHAR,YES,None,None,None
6,ionization_mode,VARCHAR,YES,None,None,None
7,instrument_type,VARCHAR,YES,None,None,None
8,precursor_mz,DOUBLE,YES,None,None,None
9,collision_energy_orig,VARCHAR,YES,None,None,None


## 2. Scale: spectra, molecules, spectra-per-molecule

In [5]:
con.execute(f"""
SELECT
  count(*) AS n_spectra,
  count(DISTINCT inchikey14) AS n_molecules,
  count(DISTINCT normalized_smiles) AS n_unique_smiles,
  count(DISTINCT inchikey) AS n_unique_inchikey
FROM '{TRAIN}'
""").fetchdf()


,n_spectra,n_molecules,n_unique_smiles,n_unique_inchikey
0,2539608,275810,277566,275950


In [6]:
sub = pd.read_csv(SUB)
test_mol = con.execute(f"SELECT DISTINCT molecule_id FROM '{TEST}'").fetchdf()
print("test spectra:", con.execute(f"SELECT count(*) FROM '{TEST}'").fetchone()[0])
print("test unique molecule_id:", test_mol['molecule_id'].nunique())
print("sample_submission rows:", len(sub))
print("molecule_id sets identical:", set(test_mol['molecule_id']) == set(sub['molecule_id']))


test spectra: 1213
test unique molecule_id: 400
sample_submission rows: 400
molecule_id sets identical: True


In [7]:
con.execute(f"""
SELECT n_spectra, count(*) AS n_molecules
FROM (SELECT inchikey14, count(*) AS n_spectra FROM '{TRAIN}' GROUP BY 1)
GROUP BY 1 ORDER BY 1 LIMIT 15
""").fetchdf()


,n_spectra,n_molecules
0,1,17033
1,2,16463
2,3,32969
3,4,43853
4,5,18741
5,6,19057
6,7,22541
7,8,32529
8,9,8803
9,10,10608


In [8]:
con.execute(f"""
SELECT molecule_id, count(*) AS n_spectra, count(DISTINCT adduct) AS n_adducts
FROM '{TEST}' GROUP BY 1 ORDER BY 2 DESC LIMIT 10
""").fetchdf()


,molecule_id,n_spectra,n_adducts
0,m_e8359d,9,3
1,m_b8415b,8,3
2,m_c63e89,8,2
3,m_0259d4,7,2
4,m_97cca9,7,2
5,m_83ac87,6,2
6,m_20b8f2,6,2
7,m_6105c1,6,2
8,m_b51204,6,2
9,m_006153,6,2


In [9]:
multi_adduct = con.execute(f"""
SELECT count(*) FROM (
  SELECT molecule_id FROM '{TEST}' GROUP BY 1 HAVING count(DISTINCT adduct) > 1
)
""").fetchone()[0]
print(f"test molecules with >1 adduct: {multi_adduct} / 400")


test molecules with >1 adduct: 108 / 400


## 3. Peak statistics

In [10]:
con.execute(f"""
SELECT min(num_peaks), median(num_peaks), avg(num_peaks), max(num_peaks)
FROM '{TRAIN}'
""").fetchdf()


,min(num_peaks),median(num_peaks),avg(num_peaks),max(num_peaks)
0,1,42.0,158.215848,73318


In [11]:
con.execute(f"""
SELECT min(num_peaks), median(num_peaks), avg(num_peaks), max(num_peaks)
FROM '{TRAIN}' WHERE instrument_type = 'timsTOF'
""").fetchdf()


,min(num_peaks),median(num_peaks),avg(num_peaks),max(num_peaks)
0,4,138.0,246.515259,10620


In [12]:
con.execute(f"""
SELECT min(len(ms2_mzs)), median(len(ms2_mzs)), avg(len(ms2_mzs)), max(len(ms2_mzs))
FROM '{TEST}'
""").fetchdf()


,min(len(ms2_mzs)),median(len(ms2_mzs)),avg(len(ms2_mzs)),max(len(ms2_mzs))
0,4,230.0,330.343776,3259


## 4. Instrument / library structure

This is the most important finding in Phase 0: `test.parquet` is 100% `timsTOF`, and in
train, `timsTOF` is (almost) synonymous with the `enveda-180` library, which is itself
almost entirely disjoint (by molecule) from every other library in train.


In [13]:
con.execute(f"""
SELECT instrument_type, count(*) n FROM '{TEST}' GROUP BY 1
""").fetchdf()


,instrument_type,n
0,timsTOF,1213


In [14]:
con.execute(f"""
SELECT ingest_lib, count(*) n, count(*) FILTER (WHERE instrument_type='timsTOF') AS n_timstof
FROM '{TRAIN}' GROUP BY 1 ORDER BY 2 DESC
""").fetchdf()


,ingest_lib,n,n_timstof
0,enveda-180,1153785,1153785
1,pluskal_ms2,527581,0
2,riken,347171,0
3,gnps,220849,0
4,massbank,101727,0
5,mona,92416,0
6,spectraverse,50933,0
7,msdial,40765,0
8,drug_plus,2545,0
9,enveda-np-examples,1184,1184


In [15]:
overlap = con.execute(f"""
WITH tims AS (SELECT DISTINCT inchikey14 FROM '{TRAIN}' WHERE instrument_type='timsTOF'),
     other AS (SELECT DISTINCT inchikey14 FROM '{TRAIN}'
               WHERE instrument_type IS DISTINCT FROM 'timsTOF' OR instrument_type IS NULL)
SELECT
  (SELECT count(*) FROM tims) AS n_tims_molecules,
  (SELECT count(*) FROM other) AS n_other_molecules,
  (SELECT count(*) FROM tims JOIN other USING(inchikey14)) AS n_overlap
""").fetchdf()
overlap


,n_tims_molecules,n_other_molecules,n_overlap
0,183191,95157,2538


In [16]:
con.execute(f"""
SELECT adduct, count(*) n FROM '{TEST}' GROUP BY 1 ORDER BY 2 DESC
""").fetchdf()


,adduct,n
0,[M+H]+,959
1,[M-H]-,193
2,[M+CH2O2-H]-,31
3,[M+Na]+,22
4,[M+NH4]+,4
5,[M+Cl]-,2
6,[M+K]+,2


## 5. Missing values

In [17]:
con.execute(f"""
SELECT
  sum((instrument_type IS NULL)::INT) AS null_instrument_type,
  sum((base_peak_intensity IS NULL)::INT) AS null_base_peak_intensity,
  sum((collision_energy_orig IS NULL)::INT) AS null_ce_orig
FROM '{TRAIN}'
""").fetchdf()


,null_instrument_type,null_base_peak_intensity,null_ce_orig
0,33053.0,1163138.0,320464.0


In [18]:
con.execute(f"""
SELECT collision_energy_orig_units, count(*) n FROM '{TRAIN}' GROUP BY 1 ORDER BY 2 DESC
""").fetchdf()


,collision_energy_orig_units,n
0,eV,1368908
1,NCE,789066
2,unknown,325078
3,V,56556


## 6. Duplicate analysis

In [19]:
con.execute(f"""
SELECT count(*) AS n_dupe_groups, sum(cnt) AS n_rows_involved FROM (
  SELECT inchikey14, adduct, precursor_mz, num_peaks, count(*) AS cnt
  FROM '{TRAIN}' GROUP BY 1,2,3,4 HAVING count(*) > 1
)
""").fetchdf()


,n_dupe_groups,n_rows_involved
0,178988,523532.0


In [20]:
con.execute(f"""
SELECT count(*) FROM (SELECT DISTINCT inchikey14, adduct FROM '{TRAIN}')
""").fetchdf()


,count_star()
0,519426


## 7. Outlier rows (resolved in the report, §12)

In [21]:
con.execute(f"""
SELECT precursor_mz, adduct, instrument_type, num_peaks
FROM '{TRAIN}' ORDER BY precursor_mz ASC LIMIT 5
""").fetchdf()


,precursor_mz,adduct,instrument_type,num_peaks
0,2.0000,[M+2H]2+,qTof,1676
1,2.0000,[M+2H]2+,qTof,2023
2,2.0000,[M+2H]2+,qTof,1660
3,32.0495,[M+H]+,None,3
4,32.0500,[M+H]+,Quattro_QQQ:10eV,3


In [22]:
con.execute(f"""
SELECT num_peaks, precursor_mz, adduct, ingest_lib, instrument_type
FROM '{TRAIN}' ORDER BY num_peaks DESC LIMIT 5
""").fetchdf()


,num_peaks,precursor_mz,adduct,ingest_lib,instrument_type
0,73318,233.092,[M+H]+,gnps,qTof
1,71296,233.093,[M+H]+,gnps,QQQ
2,71284,650.000,[M+2H]2+,gnps,qTof
3,56400,926.526,[M+H]+,mona,qTof
4,49177,633.400,[M+H]+,gnps,ESI-QTOF


## 8. Takeaways

See `research/00_dataset_report.md` for the full write-up and interpretation. Summary of
what this notebook established:

1. Real spectral-library data (GNPS/MassBank/MoNA/RIKEN/pluskal_ms2/etc. + Enveda's own
   `enveda-180` / `enveda-np-examples`), not synthetic.
2. Test is 100% timsTOF; `enveda-180` is train's only substantial timsTOF library and is
   ~97% disjoint (by molecule) from the rest of train.
3. Multi-spectrum-per-molecule (often multi-adduct) is the norm on both train and test.
4. `collision_energy_orig` mixes eV/NCE/V/unknown units — use `collision_energy_ev`.
5. A small number of rows have physically implausible `precursor_mz` or extreme
   `num_peaks` — filter/clean before relying on them.
6. `pandas`/`pyarrow` can't read these files directly here (pyarrow bug) — use `duckdb`.
